# Saving and Loading Models: Pickle, ONNX, SavedModel

## 📚 Learning Objectives

By completing this notebook, you will:
- Save models using Pickle
- Export models to ONNX format
- Save TensorFlow models as SavedModel
- Load models for inference
- Compare serialization formats

## 🔗 Prerequisites

- ✅ Understanding of model training
- ✅ TensorFlow/PyTorch knowledge
- ✅ Python knowledge

---

## Official Structure Reference

This notebook covers practical activities from **Course 11, Unit 2**:
- Saving and loading AI model using Pickle, ONNX, or SavedModel for TensorFlow
- **Source:** `DETAILED_UNIT_DESCRIPTIONS.md` - Unit 2 Practical Content

---

## Introduction

**Model serialization** is crucial for deployment. Different formats (Pickle, ONNX, SavedModel) offer different benefits for saving and loading models

## 📥 Inputs & 📤 Outputs | المدخلات والمخرجات

**Inputs:** What we use in this notebook

- Libraries and concepts as introduced in this notebook; see prerequisites and code comments.

**Outputs:** What you'll see when you run the cells

- Printed results, figures, and summaries as shown when you run the cells.

---


In [ ]:
import pickle
import numpy as np

print("✅ Libraries imported!")
print("\nSaving and Loading Models: Pickle, ONNX, SavedModel")
print("=" * 60)

print("\nSerialization Formats:")
print("  - Pickle: Python native, easy to use")
print("  - ONNX: Cross-platform, optimized")
print("  - SavedModel: TensorFlow standard")
print("  - HDF5: Keras models")
print("  - PMML: XML-based, portable")

print("\nFormat Comparison:")
print("  Pickle: Fast, Python-only, version-dependent")
print("  ONNX: Cross-platform, optimized, standard")
print("  SavedModel: TensorFlow-native, production-ready")

print("\n✅ Model serialization concepts understood!")

## 🌍 Real-World Worked Example — Export PyTorch to ONNX and Benchmark

**Industry context:**
- ONNX allows a model trained in PyTorch (research) to be deployed on NVIDIA GPUs, ARM chips, browsers (WebAssembly), or mobile (CoreML) — one format, everywhere
- Tesla's Autopilot runs ONNX models on its custom FSD chip
- Real-time translation on your phone uses ONNX Runtime

We export a trained PyTorch model to ONNX and run inference with ONNXRuntime — measuring speed difference.

In [ ]:
import torch, torch.nn as nn
import numpy as np, time

# ── Train a small model ────────────────────────────────────────────────────
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

iris = load_iris()
X = StandardScaler().fit_transform(iris.data.astype(np.float32))
y = iris.target
Xt = torch.tensor(X); Yt = torch.tensor(y, dtype=torch.long)

model = nn.Sequential(nn.Linear(4,64), nn.ReLU(), nn.Linear(64,32), nn.ReLU(), nn.Linear(32,3))
opt   = torch.optim.Adam(model.parameters())
for _ in range(300):
    loss = nn.CrossEntropyLoss()(model(Xt), Yt)
    opt.zero_grad(); loss.backward(); opt.step()
model.eval()
print("Model trained ✅")

# ── Export to ONNX ─────────────────────────────────────────────────────────
dummy = torch.randn(1, 4)
torch.onnx.export(model, dummy, '/tmp/iris.onnx',
                  input_names=['features'], output_names=['logits'],
                  dynamic_axes={'features':{0:'batch'}, 'logits':{0:'batch'}},
                  opset_version=17)
print("Exported to ONNX ✅")

# ── Run with ONNX Runtime ─────────────────────────────────────────────────
import onnxruntime as ort
sess = ort.InferenceSession('/tmp/iris.onnx', providers=['CPUExecutionProvider'])

N_RUNS = 10000
test_input = X[:50]

# PyTorch timing
start = time.perf_counter()
for _ in range(N_RUNS):
    with torch.no_grad(): model(torch.tensor(test_input))
pt_time = (time.perf_counter()-start)/N_RUNS*1000

# ONNX timing
start = time.perf_counter()
for _ in range(N_RUNS):
    sess.run(None, {'features': test_input})
onnx_time = (time.perf_counter()-start)/N_RUNS*1000

print(f"\nInference latency (50 samples, {N_RUNS} runs):")
print(f"  PyTorch:      {pt_time:.3f} ms")
print(f"  ONNX Runtime: {onnx_time:.3f} ms  (often 2-5x faster in production)")
print(f"\nSpeedup: {pt_time/onnx_time:.2f}x")
print("\nThis speedup scales dramatically for larger models — Tesla uses this for real-time autonomous driving.")

## 📚 References & Further Reading

**Docs:**
- [ONNX Official Site](https://onnx.ai/)
- [onnxruntime](https://onnxruntime.ai/) — Run ONNX on CPU/GPU/Edge
- [Netron](https://netron.app/) — Visualise ONNX model graphs

**State-of-the-Art:**
- Microsoft deploys ONNX models in Office 365 (spell-check, translation)
- NVIDIA uses TensorRT (ONNX-based) for real-time inference in autonomous vehicles
- Qualcomm chips natively accelerate ONNX models on mobile

## 📝 Summary

You learned **model packaging and serialization** — converting trained models into deployable artifacts. Pickle is simple but Python-only; ONNX is cross-platform and hardware-optimized. Production systems at Uber, Lyft, and Airbnb use ONNX and TorchScript for portability.